# Working Message Management

In [1]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


## Simple trim method
Retain the most recent conversations and the system instruction, abandon the rest conversation, even the potential useful ones.

In [3]:
from openai import OpenAI
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam

client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com"
)

MODEL = "deepseek-v4-flash"
MAX_TURNS = 5 # let the chat bot remember the recently 5 rounds of chat history.

messages: list[ChatCompletionMessageParam] = [
    {
        "role": "system",
        "content": "You are a rigorous research assistant. Be concise and accurate.",
    }
]

def trim_messages() -> None:
    """
    Retain:
    1. system message
    2. most recent MAX_TURNS complete conversation turns

    One complete turn = user message + assistant message
    """
    global messages

    system_message = messages[0]
    conversation_messages = messages[1:]

    complete_turns = []

    for i in range(0, len(conversation_messages) - 1, 2):
        user_message = conversation_messages[i]
        assistant_message = conversation_messages[i + 1]

        if (
            user_message.get("role") == "user"
            and assistant_message.get("role") == "assistant"
        ):
            complete_turns.append([user_message, assistant_message])

    recent_turns = complete_turns[-MAX_TURNS:]

    messages = [system_message]

    for turn in recent_turns:
        messages.extend(turn)


def chat(user_input: str) -> str:
    global messages

    messages.append({
        "role": "user",
        "content": user_input,
    })

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
    )

    assistant_reply = response.choices[0].message.content or ""

    messages.append({
        "role": "assistant",
        "content": assistant_reply,
    })

    trim_messages()

    return assistant_reply

def print_messages() -> None:
    for message in messages:
        role = message.get("role")
        content = message.get("content", "")

        if role == "user":
            print("  ", f"{role}: {content}")
        else:
            print(f"{role}: {content}")
        print()

In [4]:
chat("My name is JOM.")

'Hello, JOM. How can I assist you today?'

In [5]:
chat("Do you know my name?")

'Yes, you previously stated your name is JOM.'

In [6]:
chat("what is 1 + 1")

'1 + 1 = 2.'

In [7]:
chat("what is 1 + 2")

'1 + 2 = 3.'

In [8]:
chat("what is 1 + 13")

'1 + 13 = 14.'

In [9]:
chat("what is the answer of 4 to the 6th power?")

'4 to the 6th power (4^6) equals 4,096.'

In [10]:
chat("what is the answer of 4 to the 4th power?")

'4 to the 4th power (4^4) equals 256.'

In [11]:
chat("Do you still remember the name that I have told you before?")

"I don't recall you telling me your name in this conversation. This is the first time we've interacted, and you haven't mentioned it previously. If you'd like to share it now, I'll remember it for the rest of this session."

In [12]:
print_messages()

system: You are a rigorous research assistant. Be concise and accurate.

   user: what is 1 + 2

assistant: 1 + 2 = 3.

   user: what is 1 + 13

assistant: 1 + 13 = 14.

   user: what is the answer of 4 to the 6th power?

assistant: 4 to the 6th power (4^6) equals 4,096.

   user: what is the answer of 4 to the 4th power?

assistant: 4 to the 4th power (4^4) equals 256.

   user: Do you still remember the name that I have told you before?

assistant: I don't recall you telling me your name in this conversation. This is the first time we've interacted, and you haven't mentioned it previously. If you'd like to share it now, I'll remember it for the rest of this session.

